# Pipeline

In [15]:
from challenge.utils.camera import CameraDisplay
import collections
import cv2
from helperpost import npFilter_boxes, npNms, npDisplayBoxes
import numpy as np
import onnx
import onnxruntime as rt
import time
#import torch
#from typing import List

## Load TinyYOLO

In [77]:
# load onnx model
#model_path = "../models/person_only_baseline/model_best_no_bnopt.onnx"
#model_path = "../models/person_only_both_datasets_4_layers_finetuned/model_best_bnopt.onnx"
#model_path = "../models/person_only_both_datasets_4_layers_finetuned/iterative_pruning_15_epochs/model_pruned_20_bnopt.onnx"
#model_path = "../models/person_only_both_datasets_4_layers_finetuned/iterative_pruning_15_epochs/model_pruned_16_bnopt.onnx"
model_path = "../models/person_only_both_datasets_4_layers_finetuned/iterative_pruning_15_epochs_onestep/model_pruned_8_bnopt.onnx"
model = onnx.load(model_path)
#onnx.checker.check_model(model)

# create rt session
#providers = ['CUDAExecutionProvider']#,'CPUExecutionProvider']
providers = ['TensorrtExecutionProvider']
#providers = [('CUDAExecutionProvider', {"cudnn_conv_use_max_workspace": '1','cudnn_conv_algo_search': 'EXHAUSTIVE'}),'CPUExecutionProvider',]
#providers = [('TensorrtExecutionProvider', {
#                'device_id': 0,
#                'trt_max_workspace_size': 2147483648,
#                'trt_fp16_enable': False,
#            })]

sess_options = rt.SessionOptions()
#sess_options.enable_profiling = True
sess_options.graph_optimization_level = rt.GraphOptimizationLevel.ORT_ENABLE_ALL
#sess_options.optimized_model_filepath = 'optmodel.onnx'

ort_sess = rt.InferenceSession(model_path, sess_options=sess_options, providers=providers)

# Pre-allocate CUDA-accessible input/output buffers
io_binding = ort_sess.io_binding()
X_ortvalue = rt.OrtValue.ortvalue_from_numpy(
    np.zeros(ort_sess.get_inputs()[0].shape, dtype=np.float32), 'cuda', 0)
# OnnxRuntime will copy the data over to the CUDA device if 'input' is consumed by nodes on the CUDA device
io_binding.bind_input(name='input', 
                        device_type=X_ortvalue.device_name(), 
                        device_id=0, 
                        element_type=np.float32, 
                        shape=X_ortvalue.shape(), 
                        buffer_ptr=X_ortvalue.data_ptr()
                        )
io_binding.bind_output('output', 'cuda')

In [78]:
fps_history = collections.deque(maxlen=30)  
scale = 1 / 255.0   

In [85]:


now = time.time()
"""

def callback(frame):
    global now

    fps = 1/(time.time() - now)
    now = time.time()
    
    frame = frame[:320,160:480,:]
    
    model_input = scale * frame.astype('float32')
    
    # run model
    input = np.array(model_input).transpose(2,0,1).reshape(1,3,320,320)#.astype('float16')
    #print(f"reshape: {(time.time() - test)*1000:.2f} ms")
    test = time.time()
    output = ort_sess.run(['output'], {'input': input})
    inf_time = time.time() - test
    
    
    # filter boxes based on confidence score (class_score*confidence)
    output = npFilter_boxes(output[0], 0.1)
    #filter boxes based on overlap
    output = npNms(output, 0.25)
    # add bounding boxes
    frame = npDisplayBoxes(frame, output)    
    
    fps_history.append(fps)
    
    #cv2.line(frame, (125, 34), (130, 221), (255,0,0), 10) 
    cv2.putText(frame, f"fps={sum(fps_history)/len(fps_history):.2f}", (2, 25), cv2.FONT_HERSHEY_SIMPLEX, 1,
                (100, 255, 0), 2, cv2.LINE_AA)
    # cv2.putText(frame, f"inference time={inf_time*1000:.2f}", (2, 25), cv2.FONT_HERSHEY_SIMPLEX, 1,
    #            (100, 255, 0), 2, cv2.LINE_AA)
    
    return frame

"""

def callback(frame):
    global now
    global X_ortvalue
    
    frame = frame[:320,160:480,:]
    model_input = scale * frame.astype('float32')

    # inference
    t0 = time.time()            
    X_ortvalue = rt.OrtValue.ortvalue_from_numpy(
        np.asarray(model_input).transpose(2,0,1).reshape(1,3,320,320), 
        'cuda', 0) # overwrite value in GPU
    #X_ortvalue.update_inplace(input)        
    #t0 = time.time()
    ort_sess.run_with_iobinding(io_binding) # run model
    #inf_time = time.time() - t0
    
    output = io_binding.copy_outputs_to_cpu()[0] # copy output from GPU to CPU
    
    # filter boxes based on confidence score (class_score*confidence)
    output = npFilter_boxes(output, 0.3)
    #filter boxes based on overlap
    output = npNms(output, 0.25)
    # add bounding boxes
    frame = npDisplayBoxes(frame, output)    
    
    fps_history.append(1/(time.time() - now))
    #fps_history.append(inf_time * 1000)
    
    cv2.putText(frame, f"fps={sum(fps_history) / len(fps_history):.2f}", 
                (2, 25), cv2.FONT_HERSHEY_SIMPLEX, 1,
                (100, 255, 0), 2, cv2.LINE_AA)
    
    #cv2.putText(frame, "fps="+fps, (2, 25), cv2.FONT_HERSHEY_SIMPLEX, 1,
    #            (100, 255, 0), 2, cv2.LINE_AA)
    
    now = time.time()
    
    return frame


## Camera loop

In [87]:
# Initialize the camera with the callback
cam = CameraDisplay(callback)

Initializing camera...


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [88]:
# The camera stream can be started with cam.start()
# The callback gets asynchronously called (can be stopped with cam.stop())
cam.start()

In [89]:
# The camera should always be stopped and released for a new camera is instantiated (calling CameraDisplay(callback) again)
cam.stop()
cam.release()

Camera released
